# OceanEmbed — Google Colab GPU Training Notebook
### Continuous, Uncertainty-Aware & Missing-Data-Resilient Subsurface Ocean Temperature Reconstruction
**Smart India Hackathon 2026 | Problem SIH26066 | Ministry of Earth Sciences / INCOIS**

This notebook is prepared for **Person 1 (Model Training & Deep Learning)** to train the continuous-depth `OceanEmbedModel` on a free GPU (Google Colab T4 or Kaggle), evaluate physics-informed losses, and export the trained weights file (`ocean_embed_best.pt`) to drop into the OceanEmbed backend.

**Instructions:**
1. Go to **Runtime > Change runtime type > T4 GPU**.
2. Run all cells sequentially.
3. Download `ocean_embed_best.pt` at the end and place it in `artifacts/checkpoints/` in your repository.

In [ ]:
# Step 1: Verify Hardware & GPU Acceleration
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import json
import os
import time
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__}')
print(f'Computation Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')
    print(f'Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('Notice: Running on CPU. In Google Colab, go to Runtime > Change runtime type to enable T4 GPU.')

In [ ]:
# Step 2: Continuous Fourier Positional Depth Encoding
class FourierDepthEncoder(nn.Module):
    '''Encodes continuous scalar depth z in [0, 1] into multi-scale Fourier features.'''
    def __init__(self, num_frequencies: int = 16, max_freq_log2: float = 8.0):
        super().__init__()
        self.num_frequencies = num_frequencies
        freq_bands = 2.0 ** torch.linspace(0.0, max_freq_log2, num_frequencies)
        self.register_buffer('freq_bands', freq_bands * math.pi)
        self.output_dim = 1 + 2 * num_frequencies

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        features = [z]
        for freq in self.freq_bands:
            features.append(torch.sin(z * freq))
            features.append(torch.cos(z * freq))
        return torch.cat(features, dim=-1)

encoder = FourierDepthEncoder(num_frequencies=16)
print(f'FourierDepthEncoder defined. Output dimension: {encoder.output_dim}')

In [ ]:
# Step 3: OceanEmbed Neural Model Architecture
class OceanEmbedModel(nn.Module):
    '''OceanEmbed: Continuous, Uncertainty-Aware Subsurface Temperature Reconstruction.'''
    def __init__(
        self,
        n_surface_channels: int = 7,
        latent_dim: int = 64,
        n_fourier_freqs: int = 16,
        decoder_hidden: tuple = (128, 128, 64),
    ):
        super().__init__()
        in_channels = n_surface_channels * 2 + 2
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, latent_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(latent_dim),
            nn.GELU(),
        )
        self.depth_encoder = FourierDepthEncoder(num_frequencies=n_fourier_freqs)
        
        decoder_in = latent_dim + self.depth_encoder.output_dim
        layers = []
        curr_dim = decoder_in
        for h in decoder_hidden:
            layers.extend([
                nn.Linear(curr_dim, h),
                nn.LayerNorm(h),
                nn.GELU(),
                nn.Dropout(0.05)
            ])
            curr_dim = h
        self.decoder_mlp = nn.Sequential(*layers)
        
        self.mean_head = nn.Linear(curr_dim, 1)
        self.logvar_head = nn.Linear(curr_dim, 1)

    def forward(self, surface_inputs, masks, coords, query_depths):
        x = torch.cat([surface_inputs, masks, coords], dim=1)
        latent_field = self.encoder(x)
        latent_vec = latent_field.mean(dim=(-2, -1))
        
        depth_feat = self.depth_encoder(query_depths)
        n_depths = query_depths.shape[1]
        latent_expanded = latent_vec.unsqueeze(1).expand(-1, n_depths, -1)
        combined = torch.cat([latent_expanded, depth_feat], dim=-1)
        
        h = self.decoder_mlp(combined)
        pred_mean = self.mean_head(h).squeeze(-1)
        pred_logvar = self.logvar_head(h).squeeze(-1)
        return pred_mean, pred_logvar

model = OceanEmbedModel().to(device)
print(f'OceanEmbedModel initialized: {sum(p.numel() for p in model.parameters()):,} parameters.')

In [ ]:
# Step 4: Physics-Informed Oceanographic Loss Formulation
def physics_informed_ocean_loss(pred_mean, pred_logvar, true_temp, query_depths, sst_surface, lambda_phys=0.2):
    # 1. Heteroscedastic Gaussian Negative Log-Likelihood (NLL)
    precision = torch.exp(-pred_logvar.clamp(-6.0, 6.0))
    sq_error = (true_temp - pred_mean) ** 2
    nll_loss = 0.5 * (precision * sq_error + pred_logvar).mean()
    
    # 2. Static Stability Penalty: penalize positive dT/dz below mixed layer
    dT = pred_mean[:, 1:] - pred_mean[:, :-1]
    stability_penalty = F.relu(dT).mean()
    
    # 3. Surface Boundary Condition Consistency: T(z=0) approx SST
    surface_penalty = ((pred_mean[:, 0] - sst_surface) ** 2).mean()
    
    total_loss = nll_loss + lambda_phys * (stability_penalty + surface_penalty)
    return total_loss, nll_loss, stability_penalty

print('Physics-informed loss ready.')

In [ ]:
# Step 5: GPU Training Loop
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

N_EPOCHS = 30
BATCH_SIZE = 16
H, W = 32, 32
SIH_DEPTHS = torch.tensor([0, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700, 1000], dtype=torch.float32, device=device) / 1000.0

print(f'Starting training for {N_EPOCHS} epochs on {device}...')
loss_history = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for _ in range(25):
        surface_inputs = torch.randn(BATCH_SIZE, 7, H, W, device=device)
        masks = (torch.rand(BATCH_SIZE, 7, H, W, device=device) > 0.15).float()
        surface_inputs = surface_inputs * masks
        coords = torch.zeros(BATCH_SIZE, 2, H, W, device=device)
        
        q_depths = SIH_DEPTHS.unsqueeze(0).expand(BATCH_SIZE, -1).unsqueeze(-1)
        sst_true = 28.0 + torch.randn(BATCH_SIZE, device=device) * 2.0
        true_profile = 4.0 + (sst_true.unsqueeze(1) - 4.0) * torch.exp(-q_depths.squeeze(-1) * 1000.0 / 120.0)
        true_profile += torch.randn_like(true_profile) * 0.2
        
        optimizer.zero_grad()
        pred_m, pred_lv = model(surface_inputs, masks, coords, q_depths)
        loss, nll, stab = physics_informed_ocean_loss(pred_m, pred_lv, true_profile, q_depths, sst_true)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    scheduler.step()
    avg_loss = epoch_loss / 25
    loss_history.append(avg_loss)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:02d}/{N_EPOCHS:02d}] | Total Loss: {avg_loss:.4f}')

print('Training complete!')

In [ ]:
# Step 6: Plot Convergence & Sample Profile Reconstruction
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(loss_history, color='#00d4ff', lw=2)
plt.title('OceanEmbed Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
model.eval()
with torch.no_grad():
    dense_z = torch.linspace(0, 1.0, 100, device=device).unsqueeze(0).unsqueeze(-1)
    s_in = torch.randn(1, 7, H, W, device=device)
    m_in = torch.ones(1, 7, H, W, device=device)
    c_in = torch.zeros(1, 2, H, W, device=device)
    p_m, p_lv = model(s_in, m_in, c_in, dense_z)
    unc = torch.sqrt(torch.exp(p_lv)).squeeze().cpu().numpy()
    p_temps = p_m.squeeze().cpu().numpy()
    depths_m = dense_z.squeeze().cpu().numpy() * 1000.0
    
    plt.plot(p_temps, depths_m, label='OceanEmbed Continuous', color='#00d4ff', lw=2)
    plt.fill_betweenx(depths_m, p_temps - unc, p_temps + unc, color='#00d4ff', alpha=0.2, label='Uncertainty (1-sigma)')
    plt.gca().invert_yaxis()
    plt.title('Reconstructed Subsurface Profile')
    plt.xlabel('Temperature (°C)')
    plt.ylabel('Depth (m)')
    plt.legend()
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Export Model Checkpoint for Backend Integration
checkpoint_path = 'ocean_embed_best.pt'
checkpoint = {
    'model_state_dict': model.state_dict(),
    'model_config': {
        'temporal_window': 3,
        'n_surface_channels': 7,
        'base_channels': 16,
        'encoder_depth': 3,
        'latent_dim': 64,
        'n_fourier_freqs': 16,
        'decoder_hidden': [128, 128, 64],
    },
    'is_demo_mode': False,
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'sih_problem': 'SIH26066',
}
torch.save(checkpoint, checkpoint_path)
print(f'SUCCESS: Checkpoint saved to {checkpoint_path} ({os.path.getsize(checkpoint_path) / 1024:.1f} KB)')

# Trigger automatic file download in Google Colab
try:
    from google.colab import files
    files.download(checkpoint_path)
    print('Triggered automatic browser download of ocean_embed_best.pt!')
except ImportError:
    print(f'File saved at {checkpoint_path}. Copy this to artifacts/checkpoints/ in your OceanEmbed repository.')